In [1]:
# cell 1 — imports and config
import os, json, re, time, ssl, urllib.request, pickle
from pathlib import Path
from datetime import datetime
from dotenv import load_dotenv

import arxiv
import fitz
import numpy as np
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from pymongo import MongoClient
from rank_bm25 import BM25Okapi

# load .env from project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
load_dotenv(PROJECT_ROOT / '.env')

DATA_DIR   = PROJECT_ROOT / 'data'
PAPERS_DIR = DATA_DIR / 'papers'
PAPERS_DIR.mkdir(parents=True, exist_ok=True)

MONGO_URI   = os.getenv('MONGO_URI', 'mongodb://localhost:27017')
QDRANT_HOST = os.getenv('QDRANT_HOST', 'localhost')
QDRANT_PORT = int(os.getenv('QDRANT_PORT', 6333))

print('project root :', PROJECT_ROOT)
print('mongo        :', MONGO_URI)
print('qdrant       :', f'{QDRANT_HOST}:{QDRANT_PORT}')

project root : /Users/gamzeokmen/Documents/csai415-paper-rag
mongo        : mongodb://localhost:27017
qdrant       : localhost:6333


In [2]:
# cell 2 — connect to real MongoDB and real Qdrant
mongo_client = MongoClient(MONGO_URI)
db           = mongo_client['csai415_rag']
docs_col     = db['documents']
chunks_col   = db['chunks']

docs_col.create_index('doc_id', unique=True)
docs_col.create_index('doi')
chunks_col.create_index('chunk_id', unique=True)
chunks_col.create_index('doc_id')

qdrant     = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
COLLECTION = 'csai415_papers'
DIM        = 384

qdrant.recreate_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=DIM, distance=Distance.COSINE),
)

print('connected to MongoDB  — db: csai415_rag')
print(f'connected to Qdrant   — collection: {COLLECTION} (dim={DIM})')

/var/folders/xz/pslxf0wj2b7bzqx7kpqnfltr0000gn/T/ipykernel_3078/2349703847.py:16: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


connected to MongoDB  — db: csai415_rag
connected to Qdrant   — collection: csai415_papers (dim=384)


In [3]:
# cell 3 — build metadata from local PDFs (no arxiv API needed)
import fitz

existing_meta = {}
meta_path = DATA_DIR / "corpus_metadata.json"
if meta_path.exists():
    with open(meta_path) as f:
        for m in json.load(f):
            existing_meta[m["doc_id"]] = m

metadata_list = []
pdf_files = sorted(PAPERS_DIR.glob("*.pdf"))
print(f"found {len(pdf_files)} PDFs in data/papers/")

for pdf_path in pdf_files:
    doc_id = pdf_path.stem
    if doc_id in existing_meta:
        m = existing_meta[doc_id]
        metadata_list.append({
            "doc_id"     : doc_id,
            "title"      : m.get("title", doc_id),
            "authors"    : m.get("authors", []),
            "abstract"   : m.get("abstract", ""),
            "year"       : m.get("year", 2024),
            "venue"      : m.get("primary_category", "cs.IR"),
            "doi"        : "",
            "keywords"   : [m.get("primary_category", "cs.IR")],
            "status"     : "ingested",
            "ingested_at": datetime.now().isoformat(),
        })
    else:
        try:
            doc   = fitz.open(str(pdf_path))
            text  = doc[0].get_text("text")[:500]
            title = text.strip().split("\n")[0][:120]
            doc.close()
        except:
            title = doc_id
        metadata_list.append({
            "doc_id"     : doc_id,
            "title"      : title,
            "authors"    : [],
            "abstract"   : "",
            "year"       : int("20" + doc_id[:2]) if doc_id[:2].isdigit() else 2024,
            "venue"      : "cs.IR",
            "doi"        : "",
            "keywords"   : ["cs.IR"],
            "status"     : "ingested",
            "ingested_at": datetime.now().isoformat(),
        })
    print(f"  ready: {doc_id}  —  {metadata_list[-1]['title'][:50]}")

print(f"\ntotal ready: {len(metadata_list)} papers")

found 144 PDFs in data/papers/
  ready: 2002.08909  —  arXiv:2002.08909v1  [cs.CL]  10 Feb 2020
  ready: 2004.04906  —  Dense Passage Retrieval for Open-Domain Question A
  ready: 2005.11401  —  Retrieval-Augmented Generation for
  ready: 2005.14165  —  Language Models are Few-Shot Learners
  ready: 2006.15720  —  Progressive Generation of Long Text
  ready: 2007.01282  —  Leveraging Passage Retrieval with Generative Model
  ready: 2007.08124  —  LogiQA: A Challenge Dataset for Machine Reading Co
  ready: 2008.02637  —  arXiv:2008.02637v1  [cs.CL]  6 Aug 2020
  ready: 2009.03300  —  Published as a conference paper at ICLR 2021
  ready: 2009.10855  —  Controlling Style in Generated Dialogue
  ready: 2010.00904  —  Published as a conference paper at ICLR 2021
  ready: 2012.04584  —  arXiv:2012.04584v2  [cs.CL]  4 Aug 2022
  ready: 2101.00190  —  Preﬁx-Tuning: Optimizing Continuous Prompts for Ge
  ready: 2101.05779  —  Published as a conference paper at ICLR 2021
  ready: 2102.07350  —  

In [4]:
# cell 4 — PDFs already downloaded, just confirm they exist
missing = []
for meta in metadata_list:
    pdf_path = PAPERS_DIR / f"{meta['doc_id']}.pdf"
    if not pdf_path.exists():
        missing.append(meta['doc_id'])

print(f"PDFs ready  : {len(metadata_list) - len(missing)}")
print(f"PDFs missing: {len(missing)}")
if missing:
    for m in missing:
        print(f"  missing: {m}")

PDFs ready  : 144
PDFs missing: 0


In [5]:
# cell 5 — chunking helper function
def extract_and_chunk(pdf_path, doc_id, chunk_size=400, overlap=50):
    """extract text from pdf pages and split into overlapping word-window chunks"""
    chunks = []
    try:
        doc  = fitz.open(str(pdf_path))
        step = chunk_size - overlap
        for page_num, page in enumerate(doc, start=1):
            text = re.sub(r'-\n', '', page.get_text('text'))
            text = re.sub(r'\n+', ' ', text).strip()
            if len(text) < 50:
                continue
            words = text.split()
            for i, start in enumerate(range(0, max(1, len(words) - overlap), step)):
                chunk_text = ' '.join(words[start : start + chunk_size])
                if len(chunk_text) < 40:
                    continue
                chunks.append({
                    'chunk_id'   : f'{doc_id}_p{page_num}_c{i}',
                    'doc_id'     : doc_id,
                    'page_num'   : page_num,
                    'chunk_seq'  : i,
                    'chunk_type' : 'body',
                    'text'       : chunk_text,
                })
        doc.close()
    except Exception as e:
        print(f'  chunk error {doc_id}: {e}')
    return chunks

print('chunker ready')

chunker ready


In [6]:
# cell 6 — load embedding model
QUERY_PREFIX = 'Represent this sentence for searching relevant passages: '
embedder     = SentenceTransformer('BAAI/bge-small-en-v1.5')
print(f'model loaded — dim={embedder.get_embedding_dimension()}')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model loaded — dim=384


In [7]:
# cell 7 — embed and store everything into MongoDB + Qdrant
# this is the main loop — takes ~20 min for 200 papers
all_chunks = []
point_id   = 0

for meta in tqdm(metadata_list, desc='ingesting papers'):
    doc_id   = meta['doc_id']
    pdf_path = PAPERS_DIR / f'{doc_id}.pdf'

    if not pdf_path.exists():
        continue

    # chunk the pdf
    chunks = extract_and_chunk(pdf_path, doc_id)

    # add abstract as a special chunk at the front
    if meta['abstract'].strip():
        chunks.insert(0, {
            'chunk_id'   : f'{doc_id}_abstract',
            'doc_id'     : doc_id,
            'page_num'   : 0,
            'chunk_seq'  : -1,
            'chunk_type' : 'abstract',
            'text'       : meta['abstract'],
        })

    if not chunks:
        continue

    # embed all chunks for this paper
    texts = [c['text'] for c in chunks]
    vecs  = embedder.encode(
        texts,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=False
    )

    # upsert into Qdrant
    points = []
    for c, v in zip(chunks, vecs):
        points.append(PointStruct(id=point_id, vector=v.tolist(), payload=dict(c)))
        point_id += 1
    for start in range(0, len(points), 100):
        qdrant.upsert(collection_name=COLLECTION, points=points[start:start+100])

    # upsert into MongoDB chunks collection
    for c in chunks:
        chunks_col.update_one({'chunk_id': c['chunk_id']}, {'$set': c}, upsert=True)

    # upsert document metadata into MongoDB documents collection
    docs_col.update_one({'doc_id': doc_id}, {'$set': meta}, upsert=True)

    all_chunks.extend(chunks)

print(f'\ntotal chunks : {len(all_chunks)}')
print(f'qdrant points: {point_id}')
print(f'mongo docs   : {docs_col.count_documents({})}')
print(f'mongo chunks : {chunks_col.count_documents({})}')

ingesting papers:   0%|          | 0/144 [00:00<?, ?it/s]


total chunks : 6858
qdrant points: 6858
mongo docs   : 144
mongo chunks : 6858


In [8]:
# cell 8 — build BM25 index and save to disk
tokenized = [
    re.findall(r'\b[a-z]{2,}\b', c['text'].lower())
    for c in all_chunks
]
bm25 = BM25Okapi(tokenized)

bm25_path = DATA_DIR / 'bm25_index.pkl'
with open(bm25_path, 'wb') as f:
    pickle.dump({
        'bm25'      : bm25,
        'chunk_ids' : [c['chunk_id'] for c in all_chunks]
    }, f)

print(f'BM25 index built over {len(all_chunks)} chunks')
print(f'saved to {bm25_path}')

BM25 index built over 6858 chunks
saved to /Users/gamzeokmen/Documents/csai415-paper-rag/data/bm25_index.pkl


In [9]:
# cell 9 — evaluate retrieval on gold set and print metrics table
import time as _time

gold_path = DATA_DIR / 'gold_set.json'
with open(gold_path) as f:
    gold = json.load(f)

def hybrid_search(query, top_k=5):
    # dense
    qvec   = embedder.encode(QUERY_PREFIX + query, normalize_embeddings=True).tolist()
    res    = qdrant.query_points(collection_name=COLLECTION, query=qvec,
                                  limit=top_k*2, with_payload=True)
    dense  = [{'chunk_id': r.payload['chunk_id'], 'doc_id': r.payload['doc_id'],
                'score': r.score} for r in res.points]
    # bm25
    tokens = re.findall(r'\b[a-z]{2,}\b', query.lower())
    scores = bm25.get_scores(tokens)
    top_i  = np.argsort(scores)[::-1][:top_k*2]
    chunk_ids = [c['chunk_id'] for c in all_chunks]
    sparse = [{'chunk_id': chunk_ids[i], 'doc_id': all_chunks[i]['doc_id'],
               'score': float(scores[i])} for i in top_i if scores[i] > 0]
    # rrf
    rrf_scores = {}
    doc_map    = {}
    for rank, h in enumerate(dense, 1):
        rrf_scores[h['chunk_id']] = rrf_scores.get(h['chunk_id'], 0) + 1/(60+rank)
        doc_map[h['chunk_id']]    = h['doc_id']
    for rank, h in enumerate(sparse, 1):
        rrf_scores[h['chunk_id']] = rrf_scores.get(h['chunk_id'], 0) + 1/(60+rank)
        doc_map.setdefault(h['chunk_id'], h['doc_id'])
    top_chunks = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:top_k]
    return [doc_map[c] for c in top_chunks]

# run evaluation
results_table = []
latencies     = []

for g in gold:
    t0  = _time.time()
    top = hybrid_search(g['query'], top_k=5)
    latencies.append((_time.time() - t0) * 1000)
    results_table.append(g['relevant_doc'] in top)

recall = sum(results_table) / len(gold)
p95    = float(np.percentile(latencies, 95))

print(f'\n--- RETRIEVAL METRICS (gold set, n={len(gold)}) ---')
print(f'Recall@5   : {recall:.3f}  ({sum(results_table)}/{len(gold)})')
print(f'p95 latency: {p95:.1f} ms')
print('\n(copy these numbers into your D2 report)')


--- RETRIEVAL METRICS (gold set, n=10) ---
Recall@5   : 1.000  (10/10)
p95 latency: 661.2 ms

(copy these numbers into your D2 report)


In [10]:
# cell 10 — save metrics to results/
import yaml
from datetime import datetime as dt

RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

run_card = {
    'timestamp'       : dt.now().isoformat(),
    'corpus_size'     : docs_col.count_documents({}),
    'total_chunks'    : len(all_chunks),
    'embedding_model' : 'BAAI/bge-small-en-v1.5',
    'chunk_size'      : 400,
    'chunk_overlap'   : 50,
    'retrieval'       : {
        'mode'       : 'hybrid_rrf',
        'recall_at_5': round(recall, 4),
        'p95_ms'     : round(p95, 1),
    }
}

with open(RESULTS_DIR / 'd2_ingest_run_card.yaml', 'w') as f:
    yaml.dump(run_card, f, default_flow_style=False)

print('run card saved to results/d2_ingest_run_card.yaml')
print(json.dumps(run_card, indent=2))

run card saved to results/d2_ingest_run_card.yaml
{
  "timestamp": "2026-06-02T22:40:31.000955",
  "corpus_size": 144,
  "total_chunks": 6858,
  "embedding_model": "BAAI/bge-small-en-v1.5",
  "chunk_size": 400,
  "chunk_overlap": 50,
  "retrieval": {
    "mode": "hybrid_rrf",
    "recall_at_5": 1.0,
    "p95_ms": 661.2
  }
}
